## 极简可运行
下面给出一份「极简可运行」的纯 Python 版 generate 实现，只支持 greedy search 和 batch size = 1，但完整展示了 model.generate 的核心套路

- 调 model.forward 拿 logits
- 采样/取最大得 next token
- 拼回输入继续循环
- 遇到 eos_token_id 或达到 max_new_tokens 停止

你可以在此基础上再扩展 beam search、top-k、top-p、温度等策略。

## generate 本质就是 for 循环调 forward → 取/采样 next token → 拼回输入；
- 官方库把这套循环写成大状态机，并加上 KV-cache、beam、采样策略，于是成了我们看到的百行级 generate

In [ ]:
import torch
from torch import nn

class MiniGPT(nn.Module):
    def __init__(self,vocab_size=50257,d_model=512,nhead=8,num_layers=6):
        super().__init__()
        self.embed = nn.Embedding(vocab_size,d_model)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model,nhead,dim_feedforward=2048,dropout=0.1,batch_first=True
        )
        self.transformer = nn.TransformerEncoder(decoder_layer,num_layers)
        self.ln_head = nn.Linear(d_model,vocab_size)
        
    def forward(self,inputs_ids):
        # B 是批，L 是长，d 是隐藏维
        x = self.embed(input_ids) # [B,L,d]
        x = self.transformer(x)   # [B,L,d]
        return self.ln_head(x)
    @torch.no_grad()
    def generate(self,input_ids,max_new_tokens=50,eos_token_id=50256):
        self.eval()
        for _ in range(max_new_tokens):
            logits = self.forward(input_ids) # [1,L,V]
            next_token_id = logts[0:-1].argmax()
            next_token = next_token_id.view(1,1)
            if next_token_id == eos_token_id:
                break
            input_ids = torch.cat([input_ids,next_token],dim=1)
        return input_ids
# 单元测试

model = MiniGPT(vocab_size=1000)
seq = torch.randint(0,1000,(1,10))
out = model.generate(seq,max_new_tokens=20)
print("in shape:",seq.shape,"-> out shape:",out.shape)